In [1]:
from exercise_4 import get_model_config
from exercise_5 import generate_sofie_model

In [18]:
from qonnx.core.modelwrapper import ModelWrapper
import hls4ml

In [32]:
model = onnx.load("ConvWithAsymmetricPadding.onnx")

for node in model.graph.node:
    if node.op_type == "Conv":
        attr_names = [attr.name for attr in node.attribute]

        if "group" not in attr_names:
            node.attribute.append(
                helper.make_attribute("group", 1)
            )

        if "dilations" not in attr_names:
            node.attribute.append(
                helper.make_attribute("dilations", [1, 1])
            )

onnx.save(model, "model_fixed.onnx")

In [ ]:
!qonnx-cleanup model_fixed.onnx
!qonnx-to-channels-last model_fixed_clean.onnx 
!qonnx-cleanup model_fixed_clean_channels_last_clean.onnx

In [33]:
model = ModelWrapper('model_fixed_clean_channels_last_clean.onnx')

In [35]:
config = hls4ml.utils.config.config_from_onnx_model(
    model, granularity='name', backend="Vitis"
)

hls_model = hls4ml.converters.convert_from_onnx_model(
    model,
    output_dir='my-hls-test',
    hls_config=config,
)

Output layers:  ['Transpose_1']
Input shape: [1, 7, 5]
Topology:
Layer name: Transpose_0, layer type: Transpose, current shape: [[1, 1, 7, 5]]
Layer name: Conv_0, layer type: Conv, current shape: [[1, 7, 5, 1], [1, 3, 3, 1]]
Layer name: Transpose_1, layer type: Transpose, current shape: [[1, 5, 5, 1]]
Interpreting Model ...
Output layers:  ['Transpose_1']
Input shape: [1, 7, 5]
Topology:
Layer name: Transpose_0, layer type: Transpose, current shape: [[1, 1, 7, 5]]
Layer name: Conv_0, layer type: Conv, current shape: [[1, 7, 5, 1], [1, 3, 3, 1]]
Layer name: Transpose_1, layer type: Transpose, current shape: [[1, 5, 5, 1]]
Creating HLS model


In [37]:
get_model_config(hls_model)

{'model_name': 'myproject',
 'layers': [{'name': 'Conv2D_Conv_0',
   'type': 'Conv2D',
   'inputs': ['global_in'],
   'outputs': ['Conv_0_out0'],
   'attributes': {'in_width': 5,
    'out_width': 2,
    'n_chan': 1,
    'n_filt': 1,
    'pad_left': 0,
    'pad_right': 0,
    'filt_width': 3,
    'stride_width': 2,
    'dilation_width': 1,
    'in_height': 7,
    'out_height': 4,
    'pad_top': 1,
    'pad_bottom': 1,
    'filt_height': 3,
    'stride_height': 2,
    'dilation_height': 1,
    'data_format': 'channels_last',
    'weight_data': array([[[[1.]],
    
            [[1.]],
    
            [[1.]]],
    
    
           [[[1.]],
    
            [[1.]],
    
            [[1.]]],
    
    
           [[[1.]],
    
            [[1.]],
    
            [[1.]]]], dtype=float32),
    'weight_quantizer': None,
    'bias_data': array([0.]),
    'use_bias': False,
    'index': 6,
    'accum_t': <hls4ml.backends.fpga.fpga_types.HLSNamedType at 0x72e9f3bc6a50>,
    'trace': False,
    'p